# LIFE: first steps from station keeping to formation manoeuvres

This notebook implements the **first three steps and the first single-follower tracking milestone** in `Pasted markdown.md`, in the context of the preliminary LIFE semester proposal. It does not assume that example rotation periods, slew angles or error tolerances are mission requirements.

1. Reproduce one original fixed-offset follower with 7-day corrections and inspect an explicit 21-day window.
2. Generate and verify rotating and re-pointing octahedra in Python, including an animation.
3. Use Astrogator's differential corrector for a departure impulse and add an arrival impulse to reach a nearby **position and velocity**.
4. Repeat this building block for one follower over part of a rotation, plot errors and burns, and check waypoint refinement. Optional cells extend the same implementation to six spacecraft and a re-pointing case.

**How to run.** Put this notebook in the same `automation` directory as your working notebook, beside `helpers/`. First run `octahedron_constellation_pedro.ipynb` through its MCS execution cell, or open its saved, propagated scenario in STK 13. Keep that Desktop instance open. Run this notebook in your Windows **LIFEpy** kernel. Steps 1–8 need only NumPy, SciPy, Pandas, Matplotlib and Jupyter; STK begins at step 9. There is no new JWST ephemeris path to configure: Base and JWST are reused from the open scenario.

**What was established from your files.** The executable simulation configuration contains **both 28-day and 14-day** cases, despite its older introduction saying only 28 days. Their release is 6 October 2022, 07:10 UTC. It calls project-specific helpers, whose source files are not attached. The analysis reads marked CSV tables, sums burn magnitudes, subtracts fixed offsets in km, and builds time axes from an assumed 600-second cadence. Here, time comes from STK epochs; windows are explicit; moving references are subtracted at the actual sample time. Existing notebooks and existing spacecraft are preserved.

**Execution limits.** The geometry and numerical bookkeeping can be checked without STK. Windows Desktop propagation, installed component names, and the unseen helper internals must be validated on your STK installation. No STK results are pre-filled. During preparation, steps 1–8 were executed successfully; an independent free-drift circle calculation verified the coincident-burn cost and the reduction in tracking error under waypoint refinement. All 25 code cells passed Python syntax checks. The STK-dependent cells have not been executed here. Official documentation is linked at the relevant steps and collected at the end.

## 1. Imports and experiment settings


Choose a deliberately small initial experiment. `L_M` is **centre-to-vertex distance**, not edge length. The 28-day rotation and 20-degree, 2-day slew are illustrative development cases. The default dynamical run covers only 24 hours of the rotation with 6-hour waypoints. Output files go into a new timestamped directory.

The initial line of sight is specified in **ICRF axes**; the initial octahedron's +Z axis points along it. A projected ICRF +X direction defines the roll convention. Later, +Z is transported by the shortest rotation from the initial to the final line of sight; no additional twist is introduced.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from functools import lru_cache
from itertools import combinations
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from scipy.spatial.transform import Rotation

L_M = 1000.0
LOS_START = np.array([0.0, 0.0, 1.0])
LOS_END = np.array([np.sin(np.deg2rad(20)), 0.0, np.cos(np.deg2rad(20))])
ROTATION_PERIOD_S = 28 * 86400.0
SLEW_DURATION_S = 2 * 86400.0
TRACK_DURATION_S = 86400.0
WAYPOINT_S = 6 * 3600.0
SAMPLE_S = 600.0
POSITION_TOL_M = 0.10          # Endpoint numerical tolerance, not a LIFE requirement.
VELOCITY_TOL_MPS = 1e-6
BASELINE_DAYS = 21.0
START_UTCG = '6 Oct 2022 07:10:00.000'
TEMPLATE_NAME = 'Octahedron_SK14d_Xp1000m'
BASELINE_NAME = 'Octahedron_SK7d_Xp1000m'
BASELINE_OFFSET_M = 1000.0
# Set this to an existing project CSV only if your exporter saves elsewhere.
BASELINE_BURNS_CSV = None
RUN_REFINEMENT = True
RUN_SIX_SPACECRAFT = False
RUN_REPOINTING = False
RUN_FULL_ROTATION = False

OUT = Path.cwd() / 'results' / ('LIFE_first_steps_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
OUT.mkdir(parents=True, exist_ok=False)
plt.rcParams.update({'font.size': 11, 'figure.figsize': (10, 5)})
print('Outputs:', OUT)

## 2. Define an oriented octahedron


Build the six labelled vertices `(±L,0,0)`, `(0,±L,0)`, `(0,0,±L)`. Every orientation is a proper rotation. The deterministic roll convention removes the ambiguity left by specifying a line of sight alone. Antipodal re-pointing directions are rejected because the shortest rotation axis would be ambiguous.

In [ ]:
LABELS = ['Xp', 'Xm', 'Yp', 'Ym', 'Zp', 'Zm']
VERTICES = L_M * np.array([[1,0,0], [-1,0,0], [0,1,0], [0,-1,0], [0,0,1], [0,0,-1]], float)
PAIRS = list(combinations(range(6), 2))

def unit(vector):
    vector = np.asarray(vector, dtype=float)
    length = np.linalg.norm(vector)
    if length < 1e-14:
        raise ValueError('A direction must be nonzero.')
    return vector / length

def initial_orientation(los):
    z = unit(los)
    seed = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(seed, z)) > 0.95:
        seed = np.array([0.0, 1.0, 0.0])
    x = unit(seed - np.dot(seed, z) * z)
    y = np.cross(z, x)
    return np.column_stack([x, y, z])

Q0 = initial_orientation(LOS_START)
_slew_cross = np.cross(unit(LOS_START), unit(LOS_END))
SLEW_ANGLE = np.arctan2(np.linalg.norm(_slew_cross), np.dot(unit(LOS_START), unit(LOS_END)))
if np.isclose(SLEW_ANGLE, np.pi):
    raise ValueError('Choose an explicit slew axis for a 180-degree re-pointing.')
SLEW_AXIS = unit(_slew_cross) if np.linalg.norm(_slew_cross) > 1e-14 else Q0[:, 0]
print('Centre-to-vertex:', L_M, 'm; edge:', np.sqrt(2)*L_M, 'm; opposite:', 2*L_M, 'm')

## 3. Reference positions and velocities


`reference(t, operation)` returns six positions in metres and six velocities in m/s, **relative to Base with ICRF axis directions**. For rigid rotation, $\dot\rho=\omega\times\rho$. Sustained rotation has nonzero initial speed; start and stop impulses must therefore be included in the dynamical cost.

For re-pointing, use the quintic time law $s(u)=10u^3-15u^4+6u^5$. Its first and second derivatives vanish at both endpoints. The shortest-axis rotation of angle $s\alpha$ transports the whole formation and defines the final roll. This is a translational reference, independent of spacecraft attitude settings.

In [ ]:
def reference(t_s, operation='rotation'):
    t = float(t_s)
    if operation == 'rotation':
        omega = 2*np.pi / ROTATION_PERIOD_S * unit(LOS_START)
        turn = Rotation.from_rotvec(omega*t).as_matrix()
    elif operation == 'repointing':
        u = np.clip(t / SLEW_DURATION_S, 0.0, 1.0)
        s = 10*u**3 - 15*u**4 + 6*u**5
        ds = (30*u**2 - 60*u**3 + 30*u**4) / SLEW_DURATION_S
        turn = Rotation.from_rotvec(SLEW_AXIS*SLEW_ANGLE*s).as_matrix()
        omega = SLEW_AXIS*SLEW_ANGLE*ds
    elif operation == 'inertial_hold':
        turn = np.eye(3)
        omega = np.zeros(3)
    else:
        raise ValueError(operation)
    position_m = VERTICES @ (turn @ Q0).T
    velocity_mps = np.cross(omega, position_m)
    return position_m, velocity_mps

## 4. Verify geometry, velocities and endpoint conventions


Check all 15 pair distances, all six radii, the centroid, analytic velocities against centred finite differences, the rotation's periodicity, and the slew's endpoint directions and zero velocities. These checks validate the prescribed geometry; they do not test gravity or STK targeting.

In [ ]:
def pair_distances(points):
    return np.array([np.linalg.norm(points[i]-points[j]) for i,j in PAIRS])

reference_distances = pair_distances(VERTICES)
checks = []
for operation, duration in [('rotation', ROTATION_PERIOD_S), ('repointing', SLEW_DURATION_S)]:
    for t in np.linspace(0, duration, 41):
        p, v = reference(t, operation)
        np.testing.assert_allclose(np.linalg.norm(p, axis=1), L_M, atol=1e-9)
        np.testing.assert_allclose(pair_distances(p), reference_distances, atol=1e-9)
        np.testing.assert_allclose(p.mean(axis=0), 0, atol=1e-10)
        if 1 < t < duration-1:
            h = 0.25
            fd = (reference(t+h, operation)[0]-reference(t-h, operation)[0])/(2*h)
            np.testing.assert_allclose(fd, v, rtol=1e-6, atol=1e-8)
    checks.append({'operation': operation, 'geometry_and_velocity_checks': 'passed'})
np.testing.assert_allclose(reference(ROTATION_PERIOD_S)[0], reference(0)[0], atol=1e-9)
for t, los in [(0, LOS_START), (SLEW_DURATION_S, LOS_END)]:
    p, v = reference(t, 'repointing')
    np.testing.assert_allclose(unit(p[4]), unit(los), atol=1e-12)
    np.testing.assert_allclose(v, 0, atol=1e-12)
display(pd.DataFrame(checks))
print('Initial vertex speeds [m/s]:', np.linalg.norm(reference(0)[1], axis=1))

## 5. Plot and export both reference motions


Save desired position and velocity histories for every vertex. The plots use Base as origin and inertial axis directions. They describe reference geometry only. For each operation, all six trajectories use the same time parameter and orientation.

In [ ]:
fig = plt.figure(figsize=(12, 5))
for panel, (operation, duration) in enumerate([('rotation', ROTATION_PERIOD_S), ('repointing', SLEW_DURATION_S)], 1):
    times = np.linspace(0, duration, 181)
    states = [reference(t, operation) for t in times]
    positions = np.array([s[0] for s in states])
    velocities = np.array([s[1] for s in states])
    ax = fig.add_subplot(1, 2, panel, projection='3d')
    rows = []
    for j, label in enumerate(LABELS):
        ax.plot(*positions[:,j,:].T, label=label)
        ax.scatter(*positions[0,j,:], s=18)
        for k, t in enumerate(times):
            rows.append([t, label, *positions[k,j], *velocities[k,j]])
    ax.set(xlabel='ICRF X [m]', ylabel='ICRF Y [m]', zlabel='ICRF Z [m]', title=operation)
    ax.set_box_aspect((1,1,1))
    ax.legend(fontsize=8)
    pd.DataFrame(rows, columns=['elapsed_s','vertex','x_m','y_m','z_m','vx_mps','vy_mps','vz_mps']).to_csv(OUT/f'reference_{operation}.csv', index=False)
fig.tight_layout()
fig.savefig(OUT/'reference_geometry.png', dpi=180, bbox_inches='tight')
plt.show()

## 6. Animate the verified reference formation


The animation draws the 12 octahedron edges and six labelled vertices. Switch `ANIMATE_OPERATION` to `repointing` to inspect the smooth slew. A standalone HTML animation is saved and displayed; no video encoder is needed.

In [ ]:
ANIMATE_OPERATION = 'rotation'
anim_duration = ROTATION_PERIOD_S if ANIMATE_OPERATION == 'rotation' else SLEW_DURATION_S
anim_times = np.linspace(0, anim_duration, 121)
EDGES = [(i,j) for i,j in PAIRS if np.isclose(np.linalg.norm(VERTICES[i]-VERTICES[j]), np.sqrt(2)*L_M)]
fig = plt.figure(figsize=(6,6))
ax = fig.add_subplot(projection='3d')
limit = 1.2*L_M
ax.set(xlim=(-limit,limit), ylim=(-limit,limit), zlim=(-limit,limit), xlabel='ICRF X [m]', ylabel='ICRF Y [m]', zlabel='ICRF Z [m]')
ax.set_box_aspect((1,1,1))
lines = [ax.plot([], [], [], color='steelblue')[0] for _ in EDGES]
points = ax.scatter([], [], [], color='darkred')
texts = [ax.text(0,0,0,label) for label in LABELS]

def draw_frame(k):
    p, _ = reference(anim_times[k], ANIMATE_OPERATION)
    for line, (i,j) in zip(lines, EDGES):
        line.set_data_3d(*p[[i,j]].T)
    points._offsets3d = tuple(p.T)
    for label, xyz in zip(texts, p):
        label.set_position((xyz[0], xyz[1]))
        label.set_3d_properties(xyz[2], zdir='z')
    ax.set_title(f'{ANIMATE_OPERATION}: {anim_times[k]/86400:.2f} days')
    return lines

animation = FuncAnimation(fig, draw_frame, frames=len(anim_times), interval=70)
html_animation = animation.to_jshtml()
(OUT/f'reference_{ANIMATE_OPERATION}_animation.html').write_text(html_animation, encoding='utf-8')
plt.close(fig)
display(HTML(html_animation))

## 7. Define waypoint and burn bookkeeping


Always include the exact final epoch, including a shortened final interval. At an intermediate waypoint, an arrival burn and the next departure burn occur at the same epoch and in the same inertial axes. Their **vector sum** is the executable impulse. Sum the norms of these merged impulses for physical Δv; also retain the unmerged two-impulse accounting for transparency.

In [ ]:
def waypoint_times(duration_s, spacing_s):
    if duration_s <= 0 or spacing_s <= 0:
        raise ValueError('Duration and spacing must be positive.')
    times = np.arange(0.0, duration_s, spacing_s)
    return np.r_[times, duration_s]

def merge_burns(raw):
    if raw.empty:
        return pd.DataFrame(columns=['elapsed_s','dvx_mps','dvy_mps','dvz_mps','dv_mps','cumulative_mps'])
    grouped = raw.groupby('elapsed_s', sort=True)[['dvx_mps','dvy_mps','dvz_mps']].sum().reset_index()
    grouped['dv_mps'] = np.linalg.norm(grouped[['dvx_mps','dvy_mps','dvz_mps']], axis=1)
    grouped['cumulative_mps'] = grouped['dv_mps'].cumsum()
    return grouped

assert waypoint_times(10, 4).tolist() == [0,4,8,10]
trial = pd.DataFrame([[1,1,0,0], [1,-1,0,0]], columns=['elapsed_s','dvx_mps','dvy_mps','dvz_mps'])
assert merge_burns(trial)['dv_mps'].sum() == 0

## 8. Save the experiment definition


Save the input settings before propagation, so results can be traced to the selected geometry, epochs and tolerances. The inherited force-model information will be added after connecting to STK. No Δv is inferred from the animation or reference speed.

In [ ]:
settings = dict(created_utc=datetime.now(timezone.utc).isoformat(), L_m=L_M,
                los_start=LOS_START.tolist(), los_end=LOS_END.tolist(),
                rotation_period_s=ROTATION_PERIOD_S, slew_duration_s=SLEW_DURATION_S,
                track_duration_s=TRACK_DURATION_S, waypoint_s=WAYPOINT_S,
                sample_s=SAMPLE_S, position_tolerance_m=POSITION_TOL_M,
                velocity_tolerance_mps=VELOCITY_TOL_MPS, start_UTCG=START_UTCG,
                template=TEMPLATE_NAME, baseline=BASELINE_NAME)
(OUT/'settings.json').write_text(json.dumps(settings, indent=2), encoding='utf-8')

## 9. Attach to the working STK 13 scenario


Attach to the open Desktop instance using the same `agi.stk13` API as your working notebook. This does not recreate Base or change its MCS. With multiple STK instances, supply the correct process ID to `AttachToApplication` as documented by Ansys.

Use **metres and seconds throughout the Object Model** in this notebook. This makes Distance/Time and SmallDistance/Time both m/s and prevents the 1,000-fold impulse error caused by mixing their defaults. Dates become scenario **EpSec**, rather than Unix time. These preferences are shared with other notebooks attached to this instance; restore your original units before rerunning their cells. Connect command units are separate; the few retained original helper calls below explicitly run under their original km/UTCG preferences.

Sources: [Python guide](https://help.agi.com/stkdevkit/Content/python/pythonProgrammingGuide.htm), [Object Model units](https://help.agi.com/stkdevkit/Content/stkUtil/introUnitPrefs.htm).

In [ ]:
if os.name != 'nt':
    raise RuntimeError('Steps 9 onward require your Windows STK 13 Desktop / Astrogator installation.')
from agi.stk13.stkdesktop import STKDesktop
from agi.stk13.stkobjects import AgESTKObjectType, AgEVePropagatorType
from agi.stk13.stkobjects.astrogator import (
    AgEVASegmentType, AgEVAElementType, AgEVAManeuverType,
    AgEVAAttitudeControl, AgEVAControlManeuver, AgEVATargetSeqAction,
    AgEVAProfileMode,
)
stk = STKDesktop.AttachToApplication()
root = stk.Root
scenario = root.CurrentScenario
if scenario is None:
    raise RuntimeError('Open and propagate the working scenario first.')
Base = root.GetObjectFromPath('*/Satellite/Base')
template = root.GetObjectFromPath(f'*/Satellite/{TEMPLATE_NAME}')
if template.PropagatorType != AgEVePropagatorType.ePropagatorAstrogator:
    raise RuntimeError('The selected follower template must use Astrogator.')

def set_si_units():
    for dimension, value in [('Distance','m'), ('SmallDistance','m'), ('Time','sec'), ('DateFormat','EpSec')]:
        root.UnitPreferences.SetCurrentUnit(dimension, value)

set_si_units()
T0 = float(root.ConversionUtility.ConvertDate('UTCG', 'EpSec', START_UTCG))
print('Scenario:', scenario.InstanceName)
print('Start EpSec:', T0)
print('Template:', template.InstanceName, '| central body:', template.CentralBodyName)

## 10. Inspect the inherited MCS and select a coast model


List the real segment tree, including nested sequences, targeter status and propagator component names. The template's **first Propagate segment** supplies the copied coast segment. Review the printed path: if a different coast is appropriate, set `COAST_PATH` to one of the printed paths and rerun this cell. Its propagator component carries the existing integrator and force model; the notebook does not substitute two-body dynamics.

The initial spacecraft parameters are read from that propagated segment's incoming state. Inspect the named propagator in STK's Component Browser for actual gravity, SRP and integration settings. A component name alone is not proof of a force model. The unseen helpers may contain scripts, automatic sequences or changing spacecraft properties; this first experiment replaces the follower's mission sequence and uses constant, recorded spacecraft properties. It is not an exact reproduction of those hidden behaviours.

Sources: [segment collection](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSSegmentCollection.html), [Propagate](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSPropagate.html), [state](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAState.html).

In [ ]:
COAST_PATH = None

def walk_segments(collection, prefix=''):
    for i in range(collection.Count):
        segment = collection.Item(i)
        name = segment.Name
        path = f'{prefix}/{name}' if prefix else name
        yield path, segment
        if segment.Type in (AgEVASegmentType.eVASegmentTypeSequence,
                            AgEVASegmentType.eVASegmentTypeTargetSequence,
                            AgEVASegmentType.eVASegmentTypeBackwardSequence):
            yield from walk_segments(segment.Segments, path)

segments = list(walk_segments(template.Propagator.MainSequence))
for path, segment in segments:
    detail = ''
    if segment.Type == AgEVASegmentType.eVASegmentTypePropagate:
        detail = f' | propagator={segment.PropagatorName}'
    elif segment.Type == AgEVASegmentType.eVASegmentTypeTargetSequence:
        detail = ' | ' + ', '.join(str(segment.Profiles.Item(k).Status) for k in range(segment.Profiles.Count))
    print(path, detail)
coasts = [(p,s) for p,s in segments if s.Type == AgEVASegmentType.eVASegmentTypePropagate]
if not coasts:
    raise RuntimeError('No propagated coast found in the template.')
coast_path, coast_template = next((p,s) for p,s in coasts if COAST_PATH is None or p == COAST_PATH)
source_state = coast_template.InitialState
STATE_FIELDS = ['Cd','Cr','DryMass','DragArea','K1','K2','RadiationPressureArea','RadiationPressureCoeff','SRPArea',
                'FuelMass','FuelDensity','TankPressure','TankTemperature']
spacecraft_state = {key: float(getattr(source_state, key)) for key in STATE_FIELDS}
settings.update(coast_path=coast_path, propagator_component=coast_template.PropagatorName,
                spacecraft_state=spacecraft_state, spacecraft_property_units='current STK units; Distance and SmallDistance are m; Time is sec')
(OUT/'settings.json').write_text(json.dumps(settings, indent=2), encoding='utf-8')
display(pd.Series(spacecraft_state, name='inherited value'))
print('Selected coast:', coast_path, '|', coast_template.PropagatorName)

## 11. Read states with an explicit origin and frame


Use VGT to evaluate any spacecraft centre in the **same Earth-centred ICRF system**, regardless of the satellite object's central body. This avoids subtracting Sun-centred and Earth-centred data. The original baseline is also evaluated directly in `Syst_Base_L2`, including rotating-frame velocity when needed.

Endpoint states come from each **MCS segment's FinalState**, transformed to ICRF. This avoids selecting the wrong side of an impulsive discontinuity. Interior ephemeris samples are kept away from burn epochs; no spike replacement or smoothing is applied. Every VGT result must be valid, and endpoint coverage is checked before expensive runs.

Sources: [LocateInSystemWithRate](https://help.agi.com/stkdevkit/Content/DocX/AgSTKVgtLib~IAgCrdnPoint~LocateInSystemWithRate.html), [result position/velocity](https://help.agi.com/stkdevkit/Content/DocX/AgSTKVgtLib~IAgCrdnPointLocateInSystemWithRateResult.html), [GetInFrameName](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAState~GetInFrameName.html).

In [ ]:
FRAME = 'CentralBody/Earth ICRF'
earth = root.CentralBodies.Item('Earth')
sun = root.CentralBodies.Item('Sun')
icrf_system = earth.Vgt.Systems.Item('ICRF')
l2_system = sun.Vgt.Systems.Item('Syst_Base_L2')

def xyz(vector):
    return np.array([vector.X, vector.Y, vector.Z], dtype=float)

def point_state(satellite, epoch_s, system=icrf_system):
    result = satellite.Vgt.Points.Item('Center').LocateInSystemWithRate(float(epoch_s), system)
    if not result.IsValid:
        raise RuntimeError(f'No valid state for {satellite.InstanceName} at EpSec {epoch_s}. Check ephemeris coverage.')
    return xyz(result.Position), xyz(result.Velocity)

@lru_cache(maxsize=20000)
def base_state(epoch_s):
    return point_state(Base, float(epoch_s))

def segment_state(segment):
    state = segment.FinalState.GetInFrameName(FRAME)
    state.SetElementType(AgEVAElementType.eVAElementTypeCartesian)
    e = state.Element
    return np.array([e.X,e.Y,e.Z]), np.array([e.Vx,e.Vy,e.Vz])

for epoch_s in [T0, T0+BASELINE_DAYS*86400, T0+TRACK_DURATION_S, T0+SLEW_DURATION_S]:
    base_state(epoch_s)
print('Base position [m] and velocity [m/s] at start:', base_state(T0))

## 12. Reproduce one original 7-day follower


If the 7-day +X follower already exists, reuse its propagated ephemeris. Otherwise call **your original helper with its original arguments**, but create only one follower. The nested target-sequence Connect path is retained from your working notebook. This can still run beyond 21 days because the helper controls its propagation horizon; analysis is explicitly limited in the next cell.

The original repeated sequence may compute different corrections on each repetition through scripting. Therefore this baseline preserves the original run procedure, without imposing a nominal rerun that could replay the final repetition's values for every correction. A no-exception message is not a convergence certificate: inspect the original targeter and its reports. The new transfer implementation later uses separate target sequences for each interval, applies each solution, and verifies nominal endpoints.

In [ ]:
existing = {scenario.Children.Item(i).InstanceName for i in range(scenario.Children.Count)}
if BASELINE_NAME in existing:
    baseline = root.GetObjectFromPath(f'*/Satellite/{BASELINE_NAME}')
    print('Reusing:', BASELINE_NAME)
else:
    from helpers.add_satellites import add_constellation_satellite_with_position_hold
    root.UnitPreferences.SetCurrentUnit('Distance', 'km')
    root.UnitPreferences.SetCurrentUnit('DateFormat', 'UTCG')
    try:
        baseline = add_constellation_satellite_with_position_hold(
            scenario=scenario, stkRoot=root, satellite_name=BASELINE_NAME,
            leader_satellite_name='Base', separation_epoch=START_UTCG+' UTCG',
            x_offset_m=BASELINE_OFFSET_M, y_offset_m=0.0, z_offset_m=0.0,
            station_keeping_interval_days=7, verbose=True)
        command = f'Astrogator */Satellite/{BASELINE_NAME}'
        root.ExecuteCommand(command+' SetValue MainSequence.SegmentList.Sequence.SegmentList.Target_Sequence.Action Run active profiles')
        root.ExecuteCommand(command+' RunMCS')
        root.ExecuteCommand(command+' SetValue MainSequence.SegmentList.Sequence.SegmentList.Target_Sequence.Action Run nominal sequence')
    finally:
        set_si_units()
for path, segment in walk_segments(baseline.Propagator.MainSequence):
    if segment.Type == AgEVASegmentType.eVASegmentTypeTargetSequence:
        print(path, [segment.Profiles.Item(k).Status for k in range(segment.Profiles.Count)])

## 13. Plot the baseline over a real 21-day window


Sample the original follower relative to Base in its original rotating L2 axes. Subtract the requested +X offset; plot the three position components and the true vector error magnitude. The sample times are EpSec-derived elapsed days. Samples are placed inside reporting intervals to avoid burn epochs; exact start/end samples are omitted deliberately. The error shown is a **sampled** maximum, not a proven continuous maximum. Verify in STK that the targeter reaches the requested position near each 7-day correction.

In [ ]:
baseline_times = T0 + np.arange(SAMPLE_S/2, BASELINE_DAYS*86400, SAMPLE_S)
baseline_p = np.array([point_state(baseline, t, l2_system)[0] for t in baseline_times])
baseline_error = baseline_p - np.array([BASELINE_OFFSET_M,0,0])
baseline_table = pd.DataFrame({'epoch_EpSec':baseline_times, 'elapsed_s':baseline_times-T0})
for j, axis in enumerate('xyz'):
    baseline_table[f'{axis}_m'] = baseline_p[:,j]
    baseline_table[f'e{axis}_m'] = baseline_error[:,j]
baseline_table['error_m'] = np.linalg.norm(baseline_error, axis=1)
baseline_table.to_csv(OUT/'baseline_rotating_L2.csv', index=False)
fig, axes = plt.subplots(2,1, figsize=(10,7), sharex=True)
for j, label in enumerate(['X','Y','Z']):
    axes[0].plot((baseline_times-T0)/86400, baseline_error[:,j], label=label)
axes[0].set_ylabel('Offset error [m]'); axes[0].legend()
axes[1].plot((baseline_times-T0)/86400, baseline_table['error_m'])
axes[1].set(xlabel='Elapsed time [days]', ylabel='Error magnitude [m]')
for ax in axes:
    for day in [7,14,21]: ax.axvline(day, color='0.7', linestyle=':')
    ax.grid(alpha=.3)
fig.suptitle('Original 7-day position-targeting baseline, rotating L2 axes')
fig.tight_layout(); fig.savefig(OUT/'baseline_error.png', dpi=180); plt.show()
print('Sampled maximum baseline error:', baseline_table.error_m.max(), 'm')

## 14. Export and plot the original baseline burns


Use the same manoeuvre-summary exporter and parser as the supplied simulation. Read the marked CSV table, use its **actual burn timestamps**, and retain burns in the half-open interval `[start, start + 21 days)`. A burn at the final boundary belongs to the next window. The table sum is computed only for this window, rather than using a full-run header total.

The helper's timestamp-column name is not exposed by the attachments. The code recognises common start-time/epoch headings and reports all columns if it cannot resolve one; set `BASELINE_TIME_COLUMN` explicitly in that case. It never invents time from row index. The baseline is not a matched operational-cost comparison with the shorter inertial rotation experiment.

In [ ]:
from helpers.analysis import export_maneuver_summary_report, parse_maneuver_summary_report
from io import StringIO
BASELINE_TIME_COLUMN = None
style = Path.cwd()/'helpers'/'Maneuver-Summary-Detailed.rst'
root.UnitPreferences.SetCurrentUnit('Distance','km')
root.UnitPreferences.SetCurrentUnit('DateFormat','UTCG')
try:
    root.ExecuteCommand(f'ReportStyle * Load "{style.resolve()}"')
    report = export_maneuver_summary_report(root, BASELINE_NAME, verbose=False)
    parse_maneuver_summary_report(report, delete_source=False, verbose=False)
finally:
    set_si_units()
csv_path = Path(BASELINE_BURNS_CSV) if BASELINE_BURNS_CSV else Path.cwd()/'data-export'/'maneuver-summaries'/f'{BASELINE_NAME}.csv'
lines = csv_path.read_text(encoding='utf-8-sig').splitlines()
marker = next(i for i,line in enumerate(lines) if '--- BEGIN DATA ---' in line)
burns_original = pd.read_csv(StringIO('\n'.join(lines[marker+1:])))
burns_original.columns = burns_original.columns.str.strip()
if BASELINE_TIME_COLUMN is None:
    candidates = [col for col in burns_original if any(word in col.lower() for word in ['start time','start epoch','ignition','epoch'])]
    if not candidates:
        candidates = [col for col in burns_original if 'time' in col.lower() and not any(word in col.lower() for word in ['stop','end','duration'])]
    if len(candidates) != 1:
        raise ValueError(f'Set BASELINE_TIME_COLUMN to the burn-start column. Available: {list(burns_original.columns)}')
    BASELINE_TIME_COLUMN = candidates[0]

def burn_epoch(value, heading):
    text = str(value).strip()
    if 'epsec' in heading.lower(): return float(text)
    text = text.removesuffix(' UTCG').strip()
    return float(root.ConversionUtility.ConvertDate('UTCG','EpSec',text))

burns_original['epoch_EpSec'] = [burn_epoch(v, BASELINE_TIME_COLUMN) for v in burns_original[BASELINE_TIME_COLUMN]]
window = burns_original[(burns_original.epoch_EpSec >= T0) & (burns_original.epoch_EpSec < T0+BASELINE_DAYS*86400)].copy()
window['dv_mps'] = pd.to_numeric(window['Delta V (m/sec)'], errors='raise')
if window.empty or not np.isfinite(window.dv_mps).all():
    raise ValueError('No valid baseline burns in the selected window. Inspect the report and epochs.')
window = window.sort_values('epoch_EpSec')
window.to_csv(OUT/'baseline_burns_window.csv', index=False)
fig, ax = plt.subplots()
ax.stem((window.epoch_EpSec-T0)/86400, window.dv_mps)
ax.set(xlabel='Elapsed time [days]', ylabel='Burn magnitude [m/s]', title='Original baseline: individual burns')
fig.tight_layout(); fig.savefig(OUT/'baseline_burns.png', dpi=180); plt.show()
print('Window Δv:', window.dv_mps.sum(), 'm/s; burns:', len(window))

## 15. Make a separate Astrogator experiment spacecraft


Copy the existing follower object, clear **only the copy's** MCS, and create a Cartesian initial state in Earth-centred ICRF. Copy physical properties from the inspected propagated state. Each new coast is copied from the selected original segment, retaining its propagator component, but replacing its stopping conditions with one Duration condition and removing automatic-sequence triggers attached to the old stopping conditions.

This first manoeuvre study holds mass fixed (`UpdateMass=False`) and reports Δv, not propellant mass. This also makes coincident-impulse merging dynamically equivalent for mass-dependent accelerations. Attitude-dependent/custom force models or automatic sequences require review in STK before interpreting results.

Sources: [CopyObject](https://help.agi.com/stkdevkit/Content/DocX/STKObjects~IAgStkObjectCollection~CopyObject.html), [Initial State](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSInitialState.html), [spacecraft parameters](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVASpacecraftParameters.html), [InsertCopy](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSSegmentCollection~InsertCopy.html), [stopping-condition collection](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAStoppingConditionCollection.html).

In [ ]:
RUN_TAG = datetime.now().strftime('%H%M%S_%f')

def new_experiment(label, position_m, velocity_mps, start_epoch=T0):
    name = f'FS_{RUN_TAG}_{label}'
    sat = scenario.Children.CopyObject(template, name)
    driver = sat.Propagator
    driver.MainSequence.RemoveAll()
    initial = driver.MainSequence.Insert(AgEVASegmentType.eVASegmentTypeInitialState, 'Initial', '-')
    initial.CoordSystemName = FRAME
    initial.OrbitEpoch = str(float(start_epoch))
    initial.SetElementType(AgEVAElementType.eVAElementTypeCartesian)
    for attr, value in zip(['X','Y','Z','Vx','Vy','Vz'], np.r_[position_m,velocity_mps]):
        setattr(initial.Element, attr, float(value))
    mapping = {'Cd':'Cd', 'Cr':'Cr', 'DryMass':'DryMass', 'DragArea':'DragArea', 'K1':'K1','K2':'K2',
               'RadiationPressureArea':'RadiationPressureArea', 'Ck':'RadiationPressureCoeff',
               'SolarRadiationPressureArea':'SRPArea'}
    for destination, source in mapping.items():
        setattr(initial.SpacecraftParameters, destination, spacecraft_state[source])
    for attr in ['FuelMass','FuelDensity','TankPressure','TankTemperature']:
        setattr(initial.FuelTank, attr, spacecraft_state[attr])
    return sat, driver

def add_coast(sequence, name, duration_s):
    segment = sequence.InsertCopy(coast_template, '-')
    segment.Name = name
    # Original Results may refer to project-specific systems. Replace the local list.
    for i in range(segment.Results.Count-1, -1, -1):
        segment.Results.Remove(segment.Results.Item(i).Name)
    for i in range(segment.StoppingConditions.Count-1, -1, -1):
        segment.StoppingConditions.Remove(segment.StoppingConditions.Item(i).Name)
    condition = segment.StoppingConditions.Add('Duration').Properties
    condition.Trip = float(duration_s)
    condition.Tolerance = 1e-6
    segment.MinPropagationTime = 0.0
    segment.EnableMaxPropagationTime = True
    segment.MaxPropagationTime = float(duration_s)+60.0
    return segment

## 16. Configure Cartesian impulses and position targeting


A Target Sequence contains a departure impulse and a coast. The three Cartesian impulse components are the control variables, expressed in ICRF axes. The three coast-end Cartesian position components are the constraints. An arrival impulse is added **outside** that target sequence and determined from the achieved pre-arrival velocity.

The standard `Cartesian Elems/X`, `Y`, `Z` calculation objects default to Earth-centred inertial coordinates. Before targeting, the implementation compares their nominal result values with `FinalState` transformed to ICRF. It stops if the installed components use different frames or units. This is preferable to silently relying on customised component names. A 0.1 m tolerance applies to each Cartesian component; the norm bound is consequently $\sqrt{3}\,0.1$ m.

Sources: [Target Sequence controls and results](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSTargetSequence.html), [thrust-vector impulse](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAAttitudeControlImpulsiveThrustVector.html), [DC controls](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVADCControl.html), [DC results](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVADCResult.html), [Cartesian calculation objects](https://help.agi.com/stk/13.0.0/Subsystems/dataProviders/Content/html/dataProviders/Astrogator_Values.htm).

In [ ]:
def add_impulse(sequence, name, dv_mps=(0,0,0)):
    segment = sequence.Insert(AgEVASegmentType.eVASegmentTypeManeuver, name, '-')
    segment.SetManeuverType(AgEVAManeuverType.eVAManeuverTypeImpulsive)
    segment.Maneuver.SetAttitudeControlType(AgEVAAttitudeControl.eVAAttitudeControlThrustVector)
    segment.Maneuver.UpdateMass = False
    segment.Maneuver.AttitudeControl.ThrustAxesName = 'CentralBody/Earth ICRF'
    segment.Maneuver.AttitudeControl.AssignCartesian(*map(float, dv_mps))
    return segment

def make_target_leg(driver, leg_number, duration_s, desired_position_m):
    ts = driver.MainSequence.Insert(AgEVASegmentType.eVASegmentTypeTargetSequence, f'Target_{leg_number:03d}', '-')
    ts.ContinueOnFailure = False
    departure = add_impulse(ts.Segments, 'Departure')
    coast = add_coast(ts.Segments, 'Coast', duration_s)
    dc = ts.Profiles.Item('Differential Corrector')
    dc.Mode = AgEVAProfileMode.eVAProfileModeIterate
    dc.MaxIterations = 30
    dc.EnableDisplayStatus = False
    control_enums = [AgEVAControlManeuver.eVAControlManeuverImpulsiveCartesianX,
                     AgEVAControlManeuver.eVAControlManeuverImpulsiveCartesianY,
                     AgEVAControlManeuver.eVAControlManeuverImpulsiveCartesianZ]
    for axis, control_enum in zip('XYZ', control_enums):
        departure.EnableControlParameter(control_enum)
        control = dc.ControlParameters.GetControlByPaths('Departure', f'ImpulsiveMnvr.Cartesian.{axis}')
        control.Enable = True
        control.Perturbation = 1e-4       # m/s; 6 hours gives a ~2 m endpoint perturbation.
        control.MaxStep = 0.1            # m/s per iteration.
    for axis, desired in zip('XYZ', desired_position_m):
        coast.Results.Add(f'Cartesian Elems/{axis}')
        result = dc.Results.GetResultByPaths('Coast', axis)
        result.Enable = True
        result.DesiredValue = float(desired)
        result.Tolerance = POSITION_TOL_M
    ts.Action = AgEVATargetSeqAction.eVATargetSeqActionRunNominalSeq
    return ts, departure, coast, dc

## 17. Solve and verify a two-impulse leg


Run nominally first to check the calculation-object frame. Run the active differential corrector, record its status, iterations and component residuals, apply its solution, then rerun nominally. Verify the actual impulse-induced velocity jump in m/s and the achieved endpoint position.

The arrival burn is $\Delta v_a=v_\mathrm{desired}-v_\mathrm{coast,end}$. Rerun with that burn and verify both position and velocity from segment states. A failed residual, invalid epoch or unit check raises an exception and stops the experiment. A targeter status string is saved as diagnostic evidence; successful endpoint checks are required independently.

Sources: [Differential Corrector](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAProfileDifferentialCorrector.html), [profile status](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAProfile~Status.html), [MCS execution and applying solutions](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVADriverMCS.html), [GetResultValue](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSSegment~GetResultValue.html).

In [ ]:
def solve_leg(driver, number, dt, desired_r, desired_v):
    ts, departure, coast, dc = make_target_leg(driver, number, dt, desired_r)
    driver.RunMCS()
    nominal_r, _ = segment_state(coast)
    calc_r = np.array([float(coast.GetResultValue(axis)) for axis in 'XYZ'])
    np.testing.assert_allclose(calc_r, nominal_r, rtol=0, atol=0.01,
                               err_msg='Cartesian calculation objects are not in metre-valued Earth ICRF. Review them in STK.')
    ts.Action = AgEVATargetSeqAction.eVATargetSeqActionRunActiveProfiles
    driver.RunMCS()
    diagnostic = {'leg':number, 'profile_status':str(dc.Status), 'iterations':int(dc.NumIterations)}
    for axis in 'XYZ':
        result = dc.Results.GetResultByPaths('Coast', axis)
        diagnostic[f'target_residual_{axis}_m'] = float(result.CurrentValue-result.DesiredValue)
        if abs(diagnostic[f'target_residual_{axis}_m']) > POSITION_TOL_M*1.01:
            raise RuntimeError(f'Position target failed: {diagnostic}')
    ts.ApplyProfiles()
    ts.Action = AgEVATargetSeqAction.eVATargetSeqActionRunNominalSeq
    driver.RunMCS()
    r_end, v_end = segment_state(coast)
    if np.max(np.abs(r_end-desired_r)) > POSITION_TOL_M*1.05:
        raise RuntimeError('Applied nominal solution does not satisfy the position target.')
    dv_departure = np.array(departure.Maneuver.AttitudeControl.QueryCartesian(), dtype=float).ravel()
    before = departure.InitialState.GetInFrameName(FRAME)
    before.SetElementType(AgEVAElementType.eVAElementTypeCartesian)
    velocity_before = np.array([before.Element.Vx,before.Element.Vy,before.Element.Vz])
    np.testing.assert_allclose(segment_state(departure)[1]-velocity_before, dv_departure, rtol=0, atol=VELOCITY_TOL_MPS)
    dv_arrival = desired_v-v_end
    arrival = add_impulse(driver.MainSequence, f'Arrival_{number:03d}', dv_arrival)
    driver.RunMCS()
    achieved_r, achieved_v = segment_state(arrival)
    position_error = achieved_r-desired_r
    velocity_error = achieved_v-desired_v
    if np.max(np.abs(position_error)) > POSITION_TOL_M*1.05 or np.linalg.norm(velocity_error) > VELOCITY_TOL_MPS:
        raise RuntimeError('Post-arrival state verification failed.')
    expected_epoch = float(coast.InitialState.Epoch)+dt
    if abs(float(coast.FinalState.Epoch)-expected_epoch) > 1e-3:
        raise RuntimeError('Coast stopped at an unexpected epoch. Inspect stopping conditions.')
    diagnostic.update(endpoint_position_error_m=float(np.linalg.norm(position_error)),
                      pre_arrival_velocity_error_mps=float(np.linalg.norm(v_end-desired_v)),
                      endpoint_velocity_error_mps=float(np.linalg.norm(velocity_error)))
    return departure, coast, arrival, dv_departure, dv_arrival, diagnostic

## 18. Run a reference-tracking experiment


Start a follower at the correct reference position, initially **stationary relative to Base in inertial axes**. The first departure therefore pays for starting the rotation. Each later leg starts from the actual preceding arrival state; states are not reset to ideal waypoints. The final arrival matches zero relative velocity when `stop_at_end=True`, so stopping is included too.

The coast ephemeris is sampled only inside each interval. Exact initial/end coast states are added from MCS segments. Position errors are measured against the reference at those same epochs. The sampled tracking error between waypoints can be much larger than the endpoint tolerance: endpoint targeting does not constrain the whole path. Reference velocity at the final epoch may be nonzero for sustained rotation; the final endpoint velocity check uses the deliberately stopped state.

The returned tables contain raw burns, merged executable burns, coast samples, and targeter diagnostics. These files supplement your original analysis format; they are not placed into its fixed-offset CSV directories.

In [ ]:
def run_reference(label, operation, vertex=0, duration_s=TRACK_DURATION_S, spacing_s=WAYPOINT_S,
                  stop_at_end=True, start_epoch=T0):
    times = waypoint_times(duration_s, spacing_s)
    base_r0, base_v0 = base_state(float(start_epoch))
    p0, _ = reference(0, operation)
    sat, driver = new_experiment(label, base_r0+p0[vertex], base_v0, start_epoch)
    samples, burns, diagnostics = [], [], []
    for leg, (ta,tb) in enumerate(zip(times[:-1], times[1:]), 1):
        epoch_b = float(start_epoch+tb)
        br, bv = base_state(epoch_b)
        desired_p, desired_v = reference(tb, operation)
        target_v = bv + desired_v[vertex]
        if stop_at_end and leg == len(times)-1:
            target_v = bv.copy()
        departure, coast, arrival, dv0, dv1, diag = solve_leg(driver, leg, tb-ta, br+desired_p[vertex], target_v)
        diag.update(elapsed_s=float(tb), vertex=LABELS[vertex], operation=operation)
        diagnostics.append(diag)
        burns.extend([[float(ta),'departure',*dv0], [float(tb),'arrival',*dv1]])
        # Exact coast endpoints from segments; interiors from the current ephemeris.
        sample_list = [(float(ta), *segment_state(departure), 'post_departure')]
        interior = np.arange(ta+SAMPLE_S/2, tb-SAMPLE_S/4, SAMPLE_S)
        for t in interior:
            sample_list.append((float(t), *point_state(sat, start_epoch+t), 'coast'))
        sample_list.append((float(tb), *segment_state(coast), 'pre_arrival'))
        for t, actual_r, actual_v, side in sample_list:
            base_r, base_v = base_state(float(start_epoch+t))
            ref_p, ref_v = reference(t, operation)
            rel_r, rel_v = actual_r-base_r, actual_v-base_v
            error = rel_r-ref_p[vertex]
            samples.append([start_epoch+t,t,side,*rel_r,*ref_p[vertex],*rel_v,*error,np.linalg.norm(error)])
        print(f'{label}: leg {leg}/{len(times)-1}; position error {diag["endpoint_position_error_m"]:.3g} m; DC {diag["profile_status"]}')
    raw = pd.DataFrame(burns, columns=['elapsed_s','kind','dvx_mps','dvy_mps','dvz_mps'])
    raw['dv_mps'] = np.linalg.norm(raw[['dvx_mps','dvy_mps','dvz_mps']], axis=1)
    merged = merge_burns(raw)
    trajectory = pd.DataFrame(samples, columns=['epoch_EpSec','elapsed_s','side','x_m','y_m','z_m',
        'ref_x_m','ref_y_m','ref_z_m','vx_mps','vy_mps','vz_mps','ex_m','ey_m','ez_m','error_m'])
    diagnostic_table = pd.DataFrame(diagnostics)
    for suffix, table in [('trajectory',trajectory),('raw_burns',raw),('burns',merged),('targeting',diagnostic_table)]:
        table.to_csv(OUT/f'{label}_{suffix}.csv', index=False)
    summary = dict(case=label, operation=operation, vertex=LABELS[vertex], duration_s=duration_s,
                   waypoint_s=spacing_s, sampled_max_error_m=float(trajectory.error_m.max()),
                   rms_error_m=float(np.sqrt(np.mean(trajectory.error_m**2))),
                   dv_mps=float(merged.dv_mps.sum()), unmerged_dv_mps=float(raw.dv_mps.sum()),
                   first_burn_mps=float(merged.dv_mps.iloc[0]), last_burn_mps=float(merged.dv_mps.iloc[-1]))
    (OUT/f'{label}_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    return dict(satellite=sat, driver=driver, trajectory=trajectory, raw_burns=raw,
                burns=merged, targeting=diagnostic_table, summary=summary)

## 19. Demonstrate one nearby-state transfer


Run one 6-hour leg on the +X vertex's rotation trajectory. This case ends **with** the desired rotation velocity (`stop_at_end=False`), so it demonstrates arrival at a full reference state, including velocity. Initial velocity is stationary in the inertial frame relative to Base, and the departure impulse accounts for changing it. Inspect `Target_001 → Departure → Coast` and `Arrival_001` in STK.

The diagnostic table exposes the endpoint velocity mismatch **before** the arrival burn and its residual afterward. This directly tests why the old three-position target is insufficient for moving formation operations.

In [ ]:
single = run_reference('single_transfer', 'rotation', duration_s=WAYPOINT_S,
                       spacing_s=WAYPOINT_S, stop_at_end=False)
display(single['targeting'])
display(pd.Series(single['summary']))

## 20. Track part of a rotation with one follower


Use four 6-hour legs over one day. The first and final merged burns include starting and stopping the operation. At each internal waypoint, add the coincident arrival and departure vectors before computing their norm. These results are a prescribed impulsive tracking construction, not a minimum-Δv solution.

In [ ]:
rotation_case = run_reference('rotation_Xp', 'rotation')
display(pd.Series(rotation_case['summary']))

## 21. Plot the actual motion, errors and burn budget


Plot desired versus actual relative trajectories, error magnitude over actual elapsed time, the individual executable impulse magnitudes, and cumulative Δv. The desired line is evaluated densely, not joined only at the target waypoints. A small endpoint residual together with a large inter-waypoint error indicates that reference spacing needs refinement.

In [ ]:
def plot_case(case):
    tr, b = case['trajectory'], case['burns']
    operation, duration = case['summary']['operation'], case['summary']['duration_s']
    vertex = LABELS.index(case['summary']['vertex'])
    desired = np.array([reference(t, operation)[0][vertex] for t in np.linspace(0,duration,401)])
    fig = plt.figure(figsize=(12,9))
    ax = fig.add_subplot(221, projection='3d')
    ax.plot(*desired.T, '--', label='desired')
    ax.plot(tr.x_m,tr.y_m,tr.z_m, label='actual')
    ax.set(xlabel='ICRF X [m]',ylabel='ICRF Y [m]',zlabel='ICRF Z [m]'); ax.legend()
    ax = fig.add_subplot(222)
    ax.plot(tr.elapsed_s/3600, tr.error_m)
    ax.set(xlabel='Elapsed time [h]',ylabel='Position tracking error [m]'); ax.grid(alpha=.3)
    ax = fig.add_subplot(223)
    ax.stem(b.elapsed_s/3600, b.dv_mps)
    ax.set(xlabel='Elapsed time [h]',ylabel='Executable impulse [m/s]'); ax.grid(alpha=.3)
    ax = fig.add_subplot(224)
    ax.step(b.elapsed_s/3600, b.cumulative_mps, where='post')
    ax.set(xlabel='Elapsed time [h]',ylabel='Cumulative Δv [m/s]'); ax.grid(alpha=.3)
    fig.suptitle(case['summary']['case'])
    fig.tight_layout(); fig.savefig(OUT/f'{case["summary"]["case"]}_plots.png', dpi=180, bbox_inches='tight')
    plt.show()

plot_case(single)
plot_case(rotation_case)

## 22. Check waypoint refinement and a matched inertial hold


Repeat the same one-day operation with half the waypoint spacing, with the same force model, start epoch, size, tolerances, start/stop convention and reporting cadence. Compare sampled errors and executable Δv. Two spacings are a first sensitivity check, not a convergence proof; add a third spacing if conclusions change materially.

Also run a stationary **inertial** +X reference using the original spacing. This is a matched comparison for the new construction. The earlier 7-day **rotating-L2** baseline has a different orientation law, correction strategy and duration; do not subtract its full-window cost from this rotation and call the result the operation's cost.

In [ ]:
comparison_cases = [rotation_case]
if RUN_REFINEMENT:
    refined = run_reference('rotation_Xp_finer', 'rotation', spacing_s=WAYPOINT_S/2)
    comparison_cases.append(refined)
hold_case = run_reference('inertial_hold_Xp', 'inertial_hold')
comparison_cases.append(hold_case)
comparison = pd.DataFrame([case['summary'] for case in comparison_cases])
comparison.to_csv(OUT/'first_comparison.csv', index=False)
display(comparison[['case','waypoint_s','dv_mps','sampled_max_error_m','rms_error_m']])

## 23. Optional: six-spacecraft rotation and per-spacecraft expenditure


After the single-follower checks pass, enable `RUN_SIX_SPACECRAFT`. Initially retain the one-day arc. Set `RUN_FULL_ROTATION=True` only when ready to propagate a full 28-day operation: it requires many more targeter runs, and Base coverage is checked first. This implements six followers using the same state-transfer building block and produces a per-spacecraft Δv chart.

This cell does not establish collision clearance or a formation-wide continuous geometry bound. The reference is rigid, but independently targeted coast arcs can distort its actual pair separations; evaluate those jointly before selecting an operational strategy. The two vertices on the rotation axis need maintenance even though their reference rotation speeds are zero.

In [ ]:
six_cases = []
if RUN_SIX_SPACECRAFT:
    six_duration = ROTATION_PERIOD_S if RUN_FULL_ROTATION else TRACK_DURATION_S
    base_state(float(T0+six_duration))
    for vertex, label in enumerate(LABELS):
        six_cases.append(run_reference(f'six_rotation_{label}', 'rotation', vertex=vertex, duration_s=six_duration))
    per_spacecraft = pd.DataFrame([case['summary'] for case in six_cases])
    per_spacecraft.to_csv(OUT/'six_spacecraft_summary.csv', index=False)
    fig, ax = plt.subplots()
    ax.bar(per_spacecraft.vertex, per_spacecraft.dv_mps)
    ax.set(xlabel='Vertex', ylabel='Executable Δv [m/s]', title='Rotation and maintenance, including start/stop')
    fig.tight_layout(); fig.savefig(OUT/'six_spacecraft_dv.png', dpi=180); plt.show()
    display(per_spacecraft[['vertex','dv_mps','sampled_max_error_m']])
else:
    print('Optional six-spacecraft run disabled.')

## 24. Optional: first dynamical re-pointing case


Enable `RUN_REPOINTING` to follow the verified quintic slew with one +X follower. Its initial and final reference angular rates are zero, so the desired endpoints are stationary in inertial orientation. The same targeting, error plots and burn bookkeeping apply. This is a first demonstration; a full formation slew can use the six-vertex loop above with `operation="repointing"` and `duration_s=SLEW_DURATION_S`.

In [ ]:
repointing_case = None
if RUN_REPOINTING:
    repointing_case = run_reference('repointing_Xp', 'repointing', duration_s=SLEW_DURATION_S)
    plot_case(repointing_case)
    display(repointing_case['targeting'])
else:
    print('Optional dynamical re-pointing run disabled; its Python reference was verified in step 4.')

## 25. Write the run summary and restore the original notebook units


Write a compact summary of the completed cases and restore km/UTCG preferences for your original workflow. Generated experiment spacecraft remain in the open scenario for inspection; no existing spacecraft are overwritten and the scenario is not automatically saved over your original file. Use STK **Save As** if you want a separate scenario checkpoint.

Before expanding the project, inspect the inherited force model and targeter diagnostics in STK, repeat with tighter integrator/targeter settings, and sample more densely if tracking-error peaks change. Numerical endpoint tolerance, allowed operation tracking error, and integrator accuracy are different quantities. Size, line of sight and start epoch should only be varied after these checks. Changing start epoch means sampling the same Base trajectory at a new point, not designing a new halo orbit; avoid crossing a Base impulse without explicitly handling its velocity discontinuity.

In [ ]:
completed_cases = [single, *comparison_cases, *six_cases]
if repointing_case is not None:
    completed_cases.append(repointing_case)
summary = pd.DataFrame([case['summary'] for case in completed_cases])
summary.to_csv(OUT/'run_summary.csv', index=False)
root.UnitPreferences.SetCurrentUnit('Distance','km')
root.UnitPreferences.SetCurrentUnit('DateFormat','UTCG')
base_state.cache_clear()
display(summary[['case','dv_mps','sampled_max_error_m']])
print('Completed outputs:', OUT)
print('Inspect the FS_ spacecraft in STK. Save As a separate scenario if desired.')

## Interpretation and next decisions

- **Milestone 1:** `baseline_error.png`, `baseline_burns.png`, and the actual-timestamp baseline CSVs describe one original fixed-offset follower. Its initial injection, if reported at release, is included in the window total. Original helper convergence must still be inspected; the final stored repeated-targeter status does not prove that every earlier repetition converged.
- **Milestone 2:** `reference_geometry.png`, reference CSVs and the HTML animation show a verified rigid octahedron, constant-axis rotation and a smooth, convention-defined slew.
- **Milestone 3:** `single_transfer_targeting.csv` exposes the velocity error that remains after position targeting and the residual after the arrival impulse. The one-day case shows whether those transfers approximate the moving reference adequately between waypoints.
- **Cost:** `*_raw_burns.csv` retains the two-impulse construction; `*_burns.csv` merges vectors at coincident epochs. Use the merged total for an executable impulsive budget. The first/last impulse rows include start/stop effects together with local trajectory correction; they are not isolated estimates of a pure spin-up/spin-down cost.
- **Accuracy:** error maxima are sampled. Endpoint checks do not imply acceptable formation geometry throughout a coast. No fuel-balancing optimisation, minimum-fuel claim, combined-budget decomposition, scientific collision assessment or full size/orientation/halo-phase sweep is made here.

Once the first results are understood, choose a physically justified observation rotation period and tracking requirement with your supervisor. Complete one six-spacecraft rotation and one six-spacecraft slew, then vary a few sizes, directions and epochs one parameter at a time. Keep the matched inertial hold alongside the original rotating-frame baseline.

## Official Ansys documentation used

This notebook intentionally retains **`agi.stk13`**, matching the supplied code; it does not migrate to the differently named `ansys.stk.core` API. The linked SDK interfaces describe the members exposed by that wrapper.

- [Python programming guide](https://help.agi.com/stkdevkit/Content/python/pythonProgrammingGuide.htm): Desktop attachment and object access.
- [STK Object Model units](https://help.agi.com/stkdevkit/Content/stkUtil/introUnitPrefs.htm): Distance, SmallDistance, Time and DateFormat; [SetCurrentUnit](https://help.agi.com/stkdevkit/Content/DocX/STKUtil~IAgUnitPrefsDimCollection~SetCurrentUnit.html).
- [MCS driver](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVADriverMCS.html), [segments](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSSegment.html), [collections](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSSegmentCollection.html), and [InsertCopy](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSSegmentCollection~InsertCopy.html).
- [Target Sequence](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSTargetSequence.html), [Differential Corrector](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAProfileDifferentialCorrector.html), [controls](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVADCControl.html), [results](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVADCResult.html), and [status](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAProfile~Status.html).
- [Initial State](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSInitialState.html), [state transforms](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAState~GetInFrameName.html), [spacecraft parameters](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVASpacecraftParameters.html), [impulsive thrust vectors](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAAttitudeControlImpulsiveThrustVector.html), [Propagate](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAMCSPropagate.html), and [stopping conditions](https://help.agi.com/stkdevkit/Content/DocX/AgStkGatorLib~IAgVAStoppingConditionCollection.html).
- [VGT points](https://help.agi.com/stkdevkit/Content/DocX/AgSTKVgtLib~IAgCrdnPoint.html), [position and velocity evaluation](https://help.agi.com/stkdevkit/Content/DocX/AgSTKVgtLib~IAgCrdnPoint~LocateInSystemWithRate.html), and [Cartesian provider frame/origin definitions](https://help.agi.com/stk/SubSystems/dataProviders/Content/html/dataProviders/Cartesian_Position~Vehicle.htm).
- [Calculation objects](https://help.agi.com/stk/Content/gator/ab-state.htm) and [Cartesian elements](https://help.agi.com/stk/13.0.0/Subsystems/dataProviders/Content/html/dataProviders/Astrogator_Values.htm): target quantities and default coordinate systems.

The proposed science scope and first milestones come from your supplied proposal and pasted text. Settings and known helper call paths come from the two attached notebooks. The two-impulse construction, geometry functions, checks and experiment organisation are new implementation choices.